In [1]:
import torch
from transformers import AutoModel, AutoTokenizer, BertTokenizer, BertModel

import json
import os
from tqdm import tqdm
import sys


/home/adi/Dev/CaseGNN/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

model_name = 'CSHaitao/SAILER_en_finetune'
model = AutoModel.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
device = torch.device('cuda')


In [3]:

dataset = 'test'
RDIR_sum = '../DATASET/Relations-fact-' + dataset + '/result'
RDIR_refer_sen = '../DATASET/Relations-issue-' + dataset + '/result'

files = os.listdir(RDIR_sum)



In [4]:
print(len(files))

133


In [5]:

## case representation calculation
label_list = []
model.eval()
num = 0
embedding_list = []
with torch.no_grad():
    embedding_dict = {}

    for pfile in tqdm(files[:]):
        num += 1
        file_name = pfile.split('.')[0]+'.txt'
        label_list.append(file_name)
        with open(os.path.join(RDIR_sum, pfile), 'r') as f:
            original_sum_text = f.read()
            f.close()
        with open(os.path.join(RDIR_refer_sen, pfile), 'r') as file:
            original_refer_text = file.read()
            file.close()
        fact_text = "Legal facts:"+original_sum_text
        issue_text = "Legal issues:"+original_refer_text        
        cross_text = fact_text+' '+issue_text

        if num == 1:
            ## dual encoding
            fact_tokenized_id = tokenizer(fact_text, return_tensors="pt", padding=False, truncation=True, max_length=512)
            fact_embedding = model(**fact_tokenized_id)
            fact_embedding_matrix = fact_embedding[0][:,0] ##cls token embedding [1,768]                               

            issue_tokenized_id = tokenizer(issue_text, return_tensors="pt", padding=False, truncation=True, max_length=512)
            issue_embedding = model(**issue_tokenized_id)
            issue_embedding_matrix = issue_embedding[0][:,0] ##cls token embedding [1,768]             
            
            ## cross encoding
            cross_tokenized_id = tokenizer(cross_text, return_tensors="pt", padding=False, truncation=True, max_length=512)
            cross_embedding = model(**cross_tokenized_id)
            cross_embedding_matrix = cross_embedding[0][:,0] ##cls token embedding [1,768]                           
        
        else:
            fact_tokenized_id = tokenizer(fact_text, return_tensors="pt", padding=True, truncation=True, max_length=512)
            fact_embedding = model(**fact_tokenized_id)
            fact_cls_embedding = fact_embedding[0][:,0] ##cls token embedding [1,768]   
            fact_embedding_matrix = torch.cat((fact_embedding_matrix, fact_cls_embedding), 0)               
            
            issue_tokenized_id = tokenizer(issue_text, return_tensors="pt", padding=True, truncation=True, max_length=512)
            issue_embedding = model(**issue_tokenized_id)
            issue_cls_embedding = issue_embedding[0][:,0] ##cls token embedding [1,768]
            issue_embedding_matrix = torch.cat((issue_embedding_matrix,issue_cls_embedding), 0)

            cross_tokenized_id = tokenizer(cross_text, return_tensors="pt", padding=False, truncation=True, max_length=512)
            cross_embedding = model(**cross_tokenized_id)
            cross_cls_embedding = cross_embedding[0][:,0] ##cls token embedding [1,768]    
            cross_embedding_matrix = torch.cat((cross_embedding_matrix, cross_cls_embedding), 0) 


100%|██████████| 133/133 [02:00<00:00,  1.11it/s]


In [6]:

embedding_list.append([fact_embedding_matrix, issue_embedding_matrix, cross_embedding_matrix])
torch.save(embedding_list, './promptcase_embedding/'+dataset+'_fact_issue_cross_embedding.pt')


In [7]:

with open('./promptcase_embedding/'+dataset+'_fact_issue_cross_embedding_case_list.json' , "w") as fOut:
    json.dump(label_list, fOut)
    fOut.close()             

print('PromptCase embedding generation finished.')

PromptCase embedding generation finished.
